In [59]:
import pandas as pd

# Load the FINRA 2024 survey dataset
df = pd.read_csv("../data/NFCS 2024 State Data 250623.csv")

In [60]:

# Display the dataset dimensions
print("Dataset shape:", df.shape)

# Check the columns needed for our analysis
columns = ["STATEQ", "B14A_1", "wgt_s3"]

# Check for missing values
print("\nMissing values:")
print(df[columns].isna().sum())


Dataset shape: (25539, 133)

Missing values:
STATEQ    0
B14A_1    0
wgt_s3    0
dtype: int64


In [61]:

# Count responses to the investment ownership question
print("Investment ownership responses:")
print(df["B14A_1"].value_counts(dropna=False).sort_index())

# Check the number of unique state codes
print("\nNumber of unique state codes:", df["STATEQ"].nunique())

# Examine the survey weight distribution
print("\nSurvey weight summary:")
print(df["wgt_s3"].describe())


Investment ownership responses:
B14A_1
       1025
1      8933
2     14523
98      806
99      252
Name: count, dtype: int64

Number of unique state codes: 51

Survey weight summary:
count    25539.000000
mean         1.000000
std          0.272699
min          0.254169
25%          0.847721
50%          0.961990
75%          1.069004
max          6.157905
Name: wgt_s3, dtype: float64


In [62]:

# Create a copy of the dataset for analysis
investments = df.copy()

# Convert investment responses to numeric values
investments["B14A_1"] = pd.to_numeric(
    investments["B14A_1"], errors="coerce"
)

# Keep only valid Yes (1) and No (2) responses
investments = investments[
    investments["B14A_1"].isin([1, 2])
].copy()

# Display the cleaned dataset size and response counts
print("Cleaned dataset shape:", investments.shape)
print("\nValid investment responses:")
print(investments["B14A_1"].value_counts().sort_index())

# Verify the number of states remaining
print("\nStates represented:", investments["STATEQ"].nunique())


Cleaned dataset shape: (23456, 133)

Valid investment responses:
B14A_1
1.0     8933
2.0    14523
Name: count, dtype: int64

States represented: 51


In [63]:

# Create a binary investment ownership indicator
investments["owns_investments"] = (
    investments["B14A_1"] == 1
).astype(int)

# Verify the new indicator
print("Investment ownership indicator:")
print(investments["owns_investments"].value_counts())

# Confirm all state survey weights are positive
print("\nAll state weights positive:",
      (investments["wgt_s3"] > 0).all())


Investment ownership indicator:
owns_investments
0    14523
1     8933
Name: count, dtype: int64

All state weights positive: True


In [64]:

# Calculate the weighted ownership rate for each state
state_summary = (
    investments.groupby("STATEQ")
    .apply(
        lambda group: (
            (group["owns_investments"] * group["wgt_s3"]).sum()
            / group["wgt_s3"].sum()
        ) * 100,
        include_groups=False
    )
    .reset_index(name="participation_rate")
)

# Display the calculated state participation rates
print("Number of states:", len(state_summary))
print("\nState investment participation rates:")
print(state_summary.round(2).head(10))


Number of states: 51

State investment participation rates:
   STATEQ  participation_rate
0       1               30.61
1       2               39.39
2       3               33.93
3       4               29.36
4       5               42.23
5       6               42.84
6       7               43.60
7       8               38.76
8       9               53.69
9      10               40.54


In [65]:

# Display all state codes in ascending order
print("State codes:")
print(sorted(state_summary["STATEQ"].unique()))

# Check the range and number of unique codes
print("\nMinimum state code:", state_summary["STATEQ"].min())
print("Maximum state code:", state_summary["STATEQ"].max())
print("Number of unique codes:", state_summary["STATEQ"].nunique())


State codes:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51)]

Minimum state code: 1
Maximum state code: 51
Number of unique codes: 51


In [66]:

# State abbreviations in the official FINRA STATEQ order (1–51)
state_abbreviations = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "DC",
    "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY",
    "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT",
    "NE", "NV", "NH", "NJ", "NM", "NY", "NC", "ND", "OH",
    "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT",
    "VT", "VA", "WA", "WV", "WI", "WY"
]

# Full state names in the same order (used for map tooltips)
state_names = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
    "Connecticut", "Delaware", "District of Columbia", "Florida", "Georgia",
    "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky",
    "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota",
    "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
    "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota",
    "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island",
    "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
    "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"
]
assert len(state_names) == len(state_abbreviations) == 51

# Map FINRA state codes to their two-letter abbreviations
state_mapping = dict(enumerate(state_abbreviations, start=1))

state_summary["state_code"] = state_summary["STATEQ"].map(state_mapping)
state_summary["state_name"] = state_summary["STATEQ"].map(
    dict(enumerate(state_names, start=1))
)

# Verify the mapping
print("Number of mapped states:", state_summary["state_code"].nunique())
print("Missing state abbreviations:", state_summary["state_code"].isna().sum())

# Display the first 10 states and their participation rates
print("\nState investment participation rates:")
print(state_summary.head(10).round(2))


Number of mapped states: 51
Missing state abbreviations: 0

State investment participation rates:
   STATEQ  participation_rate state_code            state_name
0       1               30.61         AL               Alabama
1       2               39.39         AK                Alaska
2       3               33.93         AZ               Arizona
3       4               29.36         AR              Arkansas
4       5               42.23         CA            California
5       6               42.84         CO              Colorado
6       7               43.60         CT           Connecticut
7       8               38.76         DE              Delaware
8       9               53.69         DC  District of Columbia
9      10               40.54         FL               Florida


In [67]:

# Summarize investment participation rates across states
print("Participation rate summary:")
print(state_summary["participation_rate"].describe().round(2))

# Display the 10 states with the highest participation rates
print("\nTop 10 states:")
print(
    state_summary.nlargest(10, "participation_rate")[
        ["state_code", "participation_rate"]
    ].round(2)
)

# Display the 10 states with the lowest participation rates
print("\nBottom 10 states:")
print(
    state_summary.nsmallest(10, "participation_rate")[
        ["state_code", "participation_rate"]
    ].round(2)
)


Participation rate summary:
count    51.00
mean     36.65
std       5.28
min      26.65
25%      33.28
50%      35.87
75%      40.64
max      53.69
Name: participation_rate, dtype: float64

Top 10 states:
   state_code  participation_rate
8          DC               53.69
30         NJ               45.98
6          CT               43.60
47         WA               43.37
5          CO               42.84
20         MD               42.51
21         MA               42.36
4          CA               42.23
23         MN               41.78
34         ND               41.59

Bottom 10 states:
   state_code  participation_rate
17         KY               26.65
48         WV               27.59
18         LA               28.63
3          AR               29.36
24         MS               29.88
36         OK               30.39
0          AL               30.61
14         IN               31.15
31         NM               31.34
50         WY               31.75


In [68]:

# Divide states into three groups based on participation rates
state_summary["participation_category"] = pd.qcut(
    state_summary["participation_rate"],
    q=3,
    labels=["Low", "Moderate", "High"]
)

# Verify the number of states in each category
print("Participation categories:")
print(state_summary["participation_category"].value_counts().sort_index())

# Display the actual participation ranges for each category
print("\nParticipation rate ranges:")
print(
    state_summary.groupby(
        "participation_category", observed=True
    )["participation_rate"].agg(["min", "max"]).round(2)
)


Participation categories:
participation_category
Low         17
Moderate    17
High        17
Name: count, dtype: int64

Participation rate ranges:
                          min    max
participation_category              
Low                     26.65  34.25
Moderate                34.30  39.39
High                    39.80  53.69


In [ ]:

# Import Plotly Express for geographic visualization
import plotly.express as px

# Create a U.S. map of investment participation
fig = px.choropleth(
    state_summary,
    locations="state_code",
    locationmode="USA-states",
    color="participation_category",
    scope="usa",
    color_discrete_map={
        "Low": "#DC2626",
        "Moderate": "#8B5CF6",
        "High": "#2563EB"
    },
    category_orders={
        "participation_category": ["High", "Moderate", "Low"]
    },
    hover_name="state_name",
    hover_data={
        "state_code": False,
        "participation_category": False,
        "participation_rate": ":.1f"
    },
    labels={
        "participation_category": "Participation Level",
        "participation_rate": "Investment Ownership (%)"
    },
    title="Non-Retirement Investment Ownership by U.S. State (2024)"
)

# Add state abbreviations (FL, TX, ...) on top of each state
fig.add_scattergeo(
    locations=state_summary["state_code"],
    locationmode="USA-states",
    text=state_summary["state_code"],
    mode="text",
    textfont=dict(color="white", size=8),
    hoverinfo="skip",
    showlegend=False
)

# Note on interpretation: groups are relative, estimates are uncertain
fig.update_layout(
    title=dict(
        subtitle=dict(
            text="Groups are relative terciles; state estimates have a "
                 "margin of error of about ±4–5 percentage points."
        )
    )
)

# Display the map
fig.show()


In [ ]:

# Save the map as a static image and an interactive HTML file
fig.write_image(
    "../visualizations/ownership_map.png", width=1200, height=700, scale=2
)
fig.write_html(
    "../visualizations/ownership_map.html", include_plotlyjs="cdn"
)
print("Saved map to ../visualizations/")